# `map`, `apply`, `transform` — the full picture

Three pandas methods that look similar but behave differently. Pick using these rules:

- **`map`** → lookup / element-wise replacement. Series only (and `DataFrame.map` for cell-wise).
- **`apply`** → most flexible. Element-wise on Series; per-row or per-column on DataFrame. Output shape can change.
- **`transform`** → output **must have the same shape** as input. The killer use case: `groupby().transform()` broadcasts group stats back to the original rows.

| Method                 | Works on   | Granularity              | Output shape          | Use when…                                |
|------------------------|------------|--------------------------|-----------------------|------------------------------------------|
| `Series.map()`         | Series     | element-wise             | same length           | replace via dict / Series / fn           |
| `DataFrame.map()`*     | DataFrame  | element-wise (every cell)| same shape            | format every cell                        |
| `Series.apply()`       | Series     | element-wise             | Series or DataFrame   | extra args, fn returning Series          |
| `DataFrame.apply()`    | DataFrame  | per row OR per column    | scalar/Series/DataFrame | aggregate or transform along an axis   |
| `Series.transform()`   | Series     | element-wise             | **same shape required** | element-wise; groupby element-wise     |
| `DataFrame.transform()`| DataFrame  | per column               | **same shape required** | groupby element-wise (e.g. z-score)    |

*`DataFrame.map` replaced `DataFrame.applymap` in pandas 2.1.

## 0. Sample data used throughout

In [7]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    'name':   ['Alice', 'Bob', 'Cara', 'Dan', 'Eve'],
    'dept':   ['eng',   'eng', 'sales','sales','hr'],
    'salary': [100,     120,   90,     95,    80],
    'bonus':  [10,      15,    8,      9,     5],
})
df

,name,dept,salary,bonus
0,Alice,eng,100,10
1,Bob,eng,120,15
2,Cara,sales,90,8
3,Dan,sales,95,9
4,Eve,hr,80,5


## 1. `Series.map()` — element-wise, four flavors

Takes a **dict**, **Series**, **function**, or `na_action='ignore'`.

In [8]:
# a) dict — unmapped keys become NaN
df['dept1'] = df['dept'].map({'eng': 'Engineering', 'sales': 'Sales', 'hr': 'Human Resources'})
df

,name,dept,salary,bonus,dept1
0,Alice,eng,100,10,Engineering
1,Bob,eng,120,15,Engineering
2,Cara,sales,90,8,Sales
3,Dan,sales,95,9,Sales
4,Eve,hr,80,5,Human Resources


In [9]:
# b) Series (lookup table)
lookup = pd.Series({'eng': 0.20, 'sales': 0.15, 'hr': 0.10})
df["dep2"] = df['dept'].map(lookup)

In [10]:
df

,name,dept,salary,bonus,dept1,dep2
0,Alice,eng,100,10,Engineering,0.20
1,Bob,eng,120,15,Engineering,0.20
2,Cara,sales,90,8,Sales,0.15
3,Dan,sales,95,9,Sales,0.15
4,Eve,hr,80,5,Human Resources,0.10


In [11]:
# c) function / lambda
print(df['salary'].map(lambda x: x * 1.1).tolist())
print(df['name'].map(str.upper).tolist())

[110.00000000000001, 132.0, 99.00000000000001, 104.50000000000001, 88.0]
['ALICE', 'BOB', 'CARA', 'DAN', 'EVE']


In [12]:
# d) na_action='ignore' — leave NaN as NaN instead of looking it up
s = pd.Series(['a', None, 'b'])
print(s.map({'a': 1, 'b': 2}))                        # None → NaN via lookup miss
print(s.map({'a': 1, 'b': 2}, na_action='ignore'))    # NaN preserved without lookup

0    1.0
1    NaN
2    2.0
dtype: float64
0    1.0
1    NaN
2    2.0
dtype: float64


In [13]:
s

0       a
1    None
2       b
dtype: object

## 2. `DataFrame.map()` — same function on every cell

Replaced `DataFrame.applymap()` in pandas 2.1 (old name still works but deprecated).

In [ ]:
# Format every numeric cell as a dollar string
df[['salary', 'bonus']].map(lambda x: f'${x}')

## 3. `Series.apply()` — like `map`, but accepts extra args

If the function returns a Series, the result becomes a **DataFrame**.

In [ ]:
# Plain element-wise — same as map here
df['salary'].apply(lambda x: x * 1.1)

In [ ]:
# With extra args
def tax(x, rate):
    return x * (1 - rate)

print(df['salary'].apply(tax, args=(0.2,)).tolist())   # positional via args=
print(df['salary'].apply(tax, rate=0.2).tolist())      # keyword

In [ ]:
# Function returns a Series → result is a DataFrame
df['name'].apply(lambda s: pd.Series({'len': len(s), 'first': s[0]}))

## 4. `DataFrame.apply()` — per row or per column

```
axis=0 (default) → fn receives each COLUMN (Series of all rows)
axis=1           → fn receives each ROW    (Series of all columns)
```

In [ ]:
# a) Per column (axis=0) — collapses to one value per column
df[['salary', 'bonus']].apply(np.mean)

In [ ]:
# b) Per row (axis=1)
df.apply(lambda r: r['salary'] + r['bonus'], axis=1)

In [ ]:
# c) Row function returns a Series → DataFrame
df.apply(lambda r: pd.Series({
    'tot':   r['salary'] + r['bonus'],
    'ratio': r['bonus'] / r['salary'],
}), axis=1)

In [ ]:
# d) result_type — force the output shape
print('expand    :\n', df.apply(lambda r: [r.salary, r.bonus], axis=1, result_type='expand'), sep='')
print('\nreduce   :\n', df.apply(lambda r: [r.salary, r.bonus], axis=1, result_type='reduce'), sep='')
# 'broadcast' returns a frame with the same shape as df (values broadcast across columns)

In [ ]:
# e) Passing extra args
def grossed_up(row, factor):
    return row['salary'] * factor + row['bonus']

print(df.apply(grossed_up, axis=1, args=(1.1,)).tolist())
print(df.apply(grossed_up, axis=1, factor=1.1).tolist())

In [ ]:
# f) raw=True — pass NumPy arrays instead of Series (much faster for numeric fns)
df[['salary', 'bonus']].apply(np.sum, axis=1, raw=True)

## 5. `transform` — same shape required

Any function that aggregates (returns a scalar) will **raise** with `transform`.
Its real superpower shows up in groupby — see section 6.

In [ ]:
# a) Element-wise on a Series
df['salary'].transform(lambda x: x * 1.1)
# df['salary'].transform('mean')   # ❌ ValueError — would collapse to scalar

In [ ]:
# b) List of functions → DataFrame, one column per function
df['salary'].transform(['sqrt', np.log])

In [ ]:
# c) Per-column z-score
df[['salary', 'bonus']].transform(lambda c: (c - c.mean()) / c.std())

In [ ]:
# d) Dict — different function per column
df.transform({'salary': np.log, 'bonus': lambda x: x * 2})

## 6. `groupby().transform()` — the killer use case

Collapses group stats and **broadcasts them back to every original row.**
Compare with `agg` (1 row per group) and `apply` (whatever you return).

In [ ]:
# Average salary per department, attached back to each row
df['dept_avg']    = df.groupby('dept')['salary'].transform('mean')
df['pct_of_dept'] = df['salary'] / df['dept_avg']
df

In [ ]:
# Z-score WITHIN each department — aligned to the original index
df.groupby('dept')['salary'].transform(lambda g: (g - g.mean()) / g.std())

## 7. `agg` vs `transform` vs `apply` on groupby — side by side

```
g = df.groupby('dept')['salary']

g.agg('mean')          # 1 row per group        (collapses)
g.transform('mean')    # 1 row per ORIGINAL row (broadcasts)
g.apply(lambda x: x)   # whatever your function returns (often surprising shape)
```

In [ ]:
g = df.groupby('dept')['salary']

print('agg:\n',       g.agg('mean'),       sep='')
print('\ntransform:\n', g.transform('mean'), sep='')
print('\napply (identity):\n', g.apply(lambda x: x), sep='')

## 8. Decision flowchart

```
Is the input a Series?
├── YES → map         (lookup / replacement: dict, Series, fn)
│        apply       (extra args, fn returning Series)
│        transform   (same length required; useful in groupby)
│
└── NO (DataFrame)
    ├── Touch every cell uniformly?
    │     → DataFrame.map()      (was applymap)
    │
    ├── Function per row/column?
    │     → DataFrame.apply(fn, axis=...)
    │
    ├── Inside groupby?
    │     ├── 1 row per group?       → .agg()
    │     ├── Same-shape output?     → .transform()
    │     └── Anything else?         → .apply()
    │
    └── Built-in aggregation?
          → .agg() / .sum() / .mean() — faster than apply
```

## 9. Speed reality check

For numeric work: **vectorize first, `apply` last.**

In [ ]:
big = pd.DataFrame({'a': np.random.rand(100_000), 'b': np.random.rand(100_000)})

%timeit big.apply(lambda r: r['a'] + r['b'], axis=1)    # WORST — per-row Python
%timeit big['a'].apply(lambda x: x * 1.1)                # element-wise Python loop
%timeit big['a'] * 1.1                                   # BEST — vectorized

## Cheat sheet

```python
# map — Series only, lookup style
s.map({'a': 1, 'b': 2})
s.map(lookup_series)
s.map(fn, na_action='ignore')

# DataFrame.map — every cell (was applymap)
df.map(lambda x: f'${x}')

# apply — Series
s.apply(fn, args=(...), key=val)

# apply — DataFrame
df.apply(fn)                  # per column (axis=0)
df.apply(fn, axis=1)          # per row
df.apply(fn, axis=1, raw=True)        # pass ndarray, faster
df.apply(fn, axis=1, result_type='expand')   # Series → columns

# transform — must keep shape
s.transform(['sqrt', 'log'])
df.transform({'a': np.log, 'b': lambda x: x * 2})
df.groupby('k')['v'].transform('mean')        # broadcast group stat
df.groupby('k')['v'].transform(lambda g: (g - g.mean()) / g.std())
```

**Rules of thumb**
- Reach for **vectorized ops** (`df['a'] * 2`) before any of these.
- `map` for replacement, `apply` for flexibility, `transform` to keep shape (especially with groupby).
- `apply(..., raw=True)` is a quick win if your function works on NumPy arrays.
- Avoid `axis=1` apply on large frames — it's the slowest pattern in pandas.